# TextCNN (Convolutional Neural Network para classificação textual)

Construir uma rede neural que recebe os textos transformados em sequências de tokens, aprende representações vetoriais das palavras (embeddings) e aplica convoluções com filtros de diferentes tamanhos para identificar padrões relevantes nas respostas. Ao final, uma camada de classificação prevê uma das três classes: c1, c234 ou c5.

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

## 1. Carregamento dos dados

In [2]:
df = pd.read_csv("train_limpo.csv") # Conjunto de treino limpo

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,prezado a senhor a esclarecemos que o serviço ...,c5
1,prezada cidadã as informações sobre óbitos no ...,c1
2,prezado senhor julio a ouvidoria geral da petr...,c1
3,prezado a senhor a esclarecemos que o serviço ...,c234
4,senhor o serviço de informações ao cidadão do ...,c234


## 2. Separação dos dados

In [3]:
x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["resp_text"],
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train_text.shape)
print("Teste:", x_test_text.shape)

Treino: (16073,)
Teste: (4019,)


## 3. Importar bibliotecas e preparar rótulos

In [25]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D,
    Concatenate, Dropout, Dense
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

import os
os.environ["PYTHONHASHSEED"] = "123"

import random
SEED = 123

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

## 4. Tokenização
O vocabulário será aprendido somente com os textos de treinamento, evitando vazamento de informação.

In [26]:
MAX_WORDS = 30000
MAX_LEN = 700

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(x_train_text)

X_train_seq = tokenizer.texts_to_sequences(x_train_text)
X_test_seq = tokenizer.texts_to_sequences(x_test_text)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("Vocabulário:", vocab_size)
print("Formato treino:", X_train_pad.shape)
print("Formato teste:", X_test_pad.shape)

Vocabulário: 30000
Formato treino: (16073, 700)
Formato teste: (4019, 700)


In [27]:
comprimentos = [len(seq) for seq in X_train_seq]

print("Média:", np.mean(comprimentos))
print("Mediana:", np.median(comprimentos))
print("Percentil 75:", np.percentile(comprimentos, 75))
print("Percentil 90:", np.percentile(comprimentos, 90))
print("Percentil 95:", np.percentile(comprimentos, 95))
print("Percentil 99:", np.percentile(comprimentos, 99))

Média: 148.00603496547004
Mediana: 104.0
Percentil 75: 193.0
Percentil 90: 317.0
Percentil 95: 430.0
Percentil 99: 746.2800000000007


### Instalação da biblioteca SciKeras
Como o antigo wrapper `KerasClassifier` do próprio TensorFlow foi descontinuado, utilizamos o `scikeras`, que é o padrão atual recomendado.

In [ ]:
# !pip install scikeras

## 5. Codificação das classes

In [28]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)

y_test_encoded = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)

Classes: ['c1' 'c234' 'c5']


### Função de Criação do Modelo e Configuração do Grid Search
Vamos envelopar a arquitetura da nossa TextCNN em uma função que aceita hiperparâmetros como argumentos para que o GridSearchCV consiga testar as diferentes combinações.

In [ ]:
# 1. Corrigir incompatibilidade de versão reinstalando o scikit-learn em uma versão compatível
!pip install --upgrade "scikit-learn<1.6.0"

# Recarregar as dependências e prosseguir com o Grid Search
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

def create_textcnn_model(embedding_dim=100, num_filters=128, dropout_rate=0.5):
    input_layer = Input(shape=(MAX_LEN,))

    embedding = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    )(input_layer)

    conv_blocks = []
    for kernel_size in [3, 4, 5]:
        conv = Conv1D(
            filters=num_filters,
            kernel_size=kernel_size,
            activation="relu"
        )(embedding)
        pool = GlobalMaxPooling1D()(conv)
        conv_blocks.append(pool)

    merged = Concatenate()(conv_blocks)
    dropout = Dropout(dropout_rate)(merged)
    output = Dense(3, activation="softmax")(dropout)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Envelopando o modelo Keras com o KerasClassifier do SciKeras
clf = KerasClassifier(
    model=create_textcnn_model,
    epochs=5,
    batch_size=64,
    verbose=0,
    embedding_dim=100,
    num_filters=128,
    dropout_rate=0.5
)

# Definindo a grade de parâmetros que desejamos testar
param_grid = {
    "model__embedding_dim": [50, 100],
    "model__num_filters": [64, 128],
    "model__dropout_rate": [0.3, 0.5]
}

# Configurando o Grid Search com 3-Fold Cross Validation
grid = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,
    verbose=2
)

print("Iniciando o Grid Search com Validação Cruzada...")
grid_result = grid.fit(X_train_pad, y_train_encoded)

print(f"\nMelhor acurácia: {grid_result.best_score_} usando {grid_result.best_params_}")

Iniciando o Grid Search com Validação Cruzada...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END model__dropout_rate=0.3, model__embedding_dim=50, model__num_filters=64; total time=  36.0s
[CV] END model__dropout_rate=0.3, model__embedding_dim=50, model__num_filters=64; total time=  31.4s
[CV] END model__dropout_rate=0.3, model__embedding_dim=50, model__num_filters=64; total time=  31.6s
[CV] END model__dropout_rate=0.3, model__embedding_dim=50, model__num_filters=128; total time=  37.5s
[CV] END model__dropout_rate=0.3, model__embedding_dim=50, model__num_filters=128; total time=  38.7s
[CV] END model__dropout_rate=0.3, model__embedding_dim=50, model__num_filters=128; total time=  38.3s
[CV] END model__dropout_rate=0.3, model__embedding_dim=100, model__num_filters=64; total time=  37.7s
[CV] END model__dropout_rate=0.3, model__embedding_dim=100, model__num_filters=64; total time=  43.0s
[CV] END model__dropout_rate=0.3, model__embedding_dim=100, model__num_filters

## 6. Construção da arquitetura TextCNN
Vamos utilizar três tamanhos de filtros convolucionais (3, 4 e 5), para capturar padrões de diferentes extensões no texto.

In [8]:
embedding_dim = 50
num_filters = 64

input_layer = Input(shape=(MAX_LEN,))

embedding = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    input_length=MAX_LEN
)(input_layer)

conv_blocks = []

for kernel_size in [3, 4, 5]:

    conv = Conv1D(
        filters=num_filters,
        kernel_size=kernel_size,
        activation="relu"
    )(embedding)

    pool = GlobalMaxPooling1D()(conv)

    conv_blocks.append(pool)

merged = Concatenate()(conv_blocks)

dropout = Dropout(0.5)(merged)

output = Dense(
    3,
    activation="softmax"
)(dropout)

model = Model(
    inputs=input_layer,
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 700)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 700, 50)   │  1,500,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 698, 64)   │      9,664 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 697, 64)   │     12,864 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 696, 64)   │     16,064 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 192)       │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 3)         │        579 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,539,171 (5.87 MB)

 Trainable params: 1,539,171 (5.87 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Treinamento do modelo com os melhores hiperparâmetros

In [9]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train_encoded,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
201/201 ━━━━━━━━━━━━━━━━━━━━ 16s 38ms/step - accuracy: 0.3795 - loss: 1.0886 - val_accuracy: 0.4053 - val_loss: 1.0738
Epoch 2/15
201/201 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.4573 - loss: 1.0452 - val_accuracy: 0.4339 - val_loss: 1.0548
Epoch 3/15
201/201 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.5314 - loss: 0.9863 - val_accuracy: 0.4463 - val_loss: 1.0464
Epoch 4/15
201/201 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.6009 - loss: 0.9020 - val_accuracy: 0.4507 - val_loss: 1.0624
Epoch 5/15
201/201 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.6832 - loss: 0.7664 - val_accuracy: 0.4348 - val_loss: 1.1396
Epoch 6/15
201/201 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.7586 - loss: 0.6137 - val_accuracy: 0.4277 - val_loss: 1.2806


## 8. Avaliação no conjunto de teste

In [10]:
y_pred_prob = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_prob, axis=1)

print("Acurácia:", accuracy_score(y_test_encoded, y_pred))

print("F1-macro:", f1_score(
    y_test_encoded,
    y_pred,
    average="macro"
))

print("\nRelatório de classificação:")
print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test_encoded, y_pred))

126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Acurácia: 0.4533466036327445
F1-macro: 0.4432147081318792

Relatório de classificação:
              precision    recall  f1-score   support

          c1       0.48      0.47      0.47      1269
        c234       0.39      0.28      0.32      1371
          c5       0.47      0.61      0.53      1379

    accuracy                           0.45      4019
   macro avg       0.45      0.45      0.44      4019
weighted avg       0.45      0.45      0.44      4019


Matriz de confusão:
[[596 291 382]
 [420 378 573]
 [235 296 848]]


## 9. Comparação entre treino e teste

In [ ]:
# y_train_pred = np.argmax(
#     model.predict(X_train_pad),
#     axis=1
# )

# print("TREINO")
# print("Acurácia:", accuracy_score(y_train_encoded, y_train_pred))
# print("F1-macro:", f1_score(
#     y_train_encoded,
#     y_train_pred,
#     average="macro"
# ))

# print("\nTESTE")
# print("Acurácia:", accuracy_score(y_test_encoded, y_pred))
# print("F1-macro:", f1_score(
#     y_test_encoded,
#     y_pred,
#     average="macro"
# ))

503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
TREINO
Acurácia: 0.6331114291047097
F1-macro: 0.6258389420830586

TESTE
Acurácia: 0.45857178402587706
F1-macro: 0.44641511036993703


## 10. Treinamento do Modelo Final e Predição no Conjunto de Teste Extra (`test1_limpo.csv`)

Nesta etapa, vamos recriar e treinar o nosso modelo com os melhores hiperparâmetros encontrados pelo Grid Search usando o conjunto completo de treino, carregar o arquivo de teste externo, aplicar as transformações necessárias e gerar as predições rotuladas.

In [29]:
# 1. Definir o modelo com os melhores parâmetros encontrados
best_embedding_dim = 50
best_num_filters = 64
best_dropout_rate = 0.5

final_input = Input(shape=(MAX_LEN,))

final_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=best_embedding_dim,
    input_length=MAX_LEN
)(final_input)

final_conv_blocks = []
for kernel_size in [3, 4, 5]:
    conv = Conv1D(
        filters=best_num_filters,
        kernel_size=kernel_size,
        activation="relu"
    )(final_embedding)
    pool = GlobalMaxPooling1D()(conv)
    final_conv_blocks.append(pool)

final_merged = Concatenate()(final_conv_blocks)
final_dropout = Dropout(best_dropout_rate)(final_merged)
final_output = Dense(3, activation="softmax")(final_dropout)

final_model = Model(inputs=final_input, outputs=final_output)
final_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 2. Treinar o modelo final com todo o conjunto de treino
# Usamos um pequeno split de validação apenas para o EarlyStopping monitorar o overfitting
early_stopping_final = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

print("Treinando o modelo definitivo...")
final_model.fit(
    X_train_pad,
    y_train_encoded,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping_final],
    verbose=1
)

Treinando o modelo definitivo...
Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


227/227 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.3815 - loss: 1.0865 - val_accuracy: 0.4148 - val_loss: 1.0732
Epoch 2/10
227/227 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.4648 - loss: 1.0412 - val_accuracy: 0.4241 - val_loss: 1.0539
Epoch 3/10
227/227 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.5289 - loss: 0.9847 - val_accuracy: 0.4453 - val_loss: 1.0489
Epoch 4/10
227/227 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.6030 - loss: 0.8973 - val_accuracy: 0.4502 - val_loss: 1.0733
Epoch 5/10
227/227 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.6763 - loss: 0.7710 - val_accuracy: 0.4434 - val_loss: 1.1443
Epoch 6/10
227/227 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.7470 - loss: 0.6283 - val_accuracy: 0.4353 - val_loss: 1.2758


In [30]:
# 3. Carregar o arquivo test1_limpo.csv
df_test1 = pd.read_csv("test1_limpo.csv")
print("Formato do arquivo test1_limpo.csv:", df_test1.shape)

# 4. Tokenizar e aplicar padding usando o MESMO tokenizer ajustado no treino
X_test1_seq = tokenizer.texts_to_sequences(df_test1["resp_text"])
X_test1_pad = pad_sequences(
    X_test1_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

# 5. Realizar as predições com o modelo definitivo
print("Gerando as predições...")
y_test1_pred_prob = final_model.predict(X_test1_pad)
y_test1_pred_numeric = np.argmax(y_test1_pred_prob, axis=1)

# 6. Converter os índices numéricos de volta para as labels de strings ('c1', 'c234', 'c5')
y_test1_pred_labels = label_encoder.inverse_transform(y_test1_pred_numeric)

# 7. Adicionar as predições ao DataFrame original e salvar em um novo arquivo CSV
df_test1["clarity_predicted"] = y_test1_pred_labels
df_test1.to_csv("test1_rotulado_textcnn.csv", index=False)

print("Predições concluídas e salvas com sucesso em 'test1_rotulado_textcnn.csv'!")
display(df_test1.head())

Formato do arquivo test1_limpo.csv: (900, 2)
Gerando as predições...
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Predições concluídas e salvas com sucesso em 'test1_rotulado_textcnn.csv'!


,resp_text,clarity,clarity_predicted
0,prezada hellen dentre as pesquisas domiciliare...,c5,c234
1,senhor francisco o serviço de informações ao c...,c1,c234
2,prezada srª raylane em resposta a sua solicita...,c5,c5
3,prezada senhora de acordo com o art 13 do decr...,c1,c234
4,prezado a senhor a em atendimento ao pedido de...,c234,c1


## 11. Comparação e Avaliação de Performance com o Gabarito (Teste Estatístico)

Vamos ler o arquivo com as classificações ideais (`test1-rotulado-para-teste-estatistico.xlsx`), alinhar com as previsões obtidas do modelo (`test1_rotulado_textcnn.csv`) e calcular as métricas oficiais de desempenho.

In [31]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# 1. Carregar os dois arquivos diretamente
df_gabarito = pd.read_excel("test1-rotulado-para-teste-estatistico.xlsx")
df_predito = pd.read_csv("test1_rotulado_textcnn.csv")

# 2. Extrair diretamente as colunas
y_real = df_gabarito["clarity"]
y_predito_final = df_predito["clarity_predicted"]

print(f"Total de registros no Gabarito: {len(y_real)}")
print(f"Total de registros no Predito: {len(y_predito_final)}")

# 3. Calcular e exibir as métricas de performance sem merge
acuracia = accuracy_score(y_real, y_predito_final)
f1_macro = f1_score(y_real, y_predito_final, average="macro")

print(f"\n======== RESULTADOS NO CONJUNTO TESTE 1 ========")
print(f"Acurácia Final: {acuracia:.4f}")
print(f"F1-Macro Final: {f1_macro:.4f}")

print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_real, y_predito_final))

print("\nMatriz de Confusão:")
print(confusion_matrix(y_real, y_predito_final, labels=list(np.unique(y_real))))

Total de registros no Gabarito: 900
Total de registros no Predito: 900

======== RESULTADOS NO CONJUNTO TESTE 1 ========
Acurácia Final: 0.4711
F1-Macro Final: 0.4682

Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

          c1       0.49      0.49      0.49       300
        c234       0.42      0.36      0.39       300
          c5       0.49      0.56      0.52       300

    accuracy                           0.47       900
   macro avg       0.47      0.47      0.47       900
weighted avg       0.47      0.47      0.47       900


Matriz de Confusão:
[[148  83  69]
 [ 88 109 103]
 [ 65  68 167]]
